In [ ]:
# https://colab.research.google.com/github/<user>/<repo>/blob/<branch>/<path-to-notebook>.ipynb

In [ ]:
!apt-get -y install ghostscript python3-tk
!pip install -q camelot-py[cv] pandas openpyxl

In [ ]:
import os

if not os.path.exists("fg_interactive_budget"):
    !git clone https://github.com/abelunbound/fg_interactive_budget.git
%cd fg_interactive_budget

In [ ]:
# Colab watcher/orchestrator for your 4-step ETL chain
# Runs each script only after its required input file exists.

import time
import subprocess
from pathlib import Path
from datetime import datetime

# ---------------------------
# Config
# ---------------------------
POLL_SECONDS = 5
TIMEOUT_SECONDS = 60 * 60  # 1 hour max wait per file (set None for no timeout)

# File dependencies (exactly matching your scripts)
INPUT_PDF = Path("data_approved/inputs/2026 Appropriation Bill Details.pdf")
OUT_01 = Path("data_approved/outputs/output.xlsx")
OUT_02 = Path("data_approved/outputs/output_merged.xlsx")
OUT_03 = Path("data_approved/outputs/output_merged_with_mda.xlsx")
OUT_04 = Path("data_approved/outputs/budget_split_into_capital_and_all_envelopes.xlsx")

PIPELINE = [
    {
        "name": "_01_pdf_to_excel_converter",
        "wait_for": INPUT_PDF,
        "run": "python data/etl_pipeline/_01_pdf_to_excel_converter.py",
        "expect": OUT_01,
    },
    {
        "name": "_02_merge_sheets",
        "wait_for": OUT_01,
        "run": "python data/etl_pipeline/_02_merge_sheets.py",
        "expect": OUT_02,
    },
    {
        "name": "_03_add_agency_identifiers",
        "wait_for": OUT_02,
        "run": "python data/etl_pipeline/_03_add_agency_identifiers.py",
        "expect": OUT_03,
    },
    {
        "name": "_04_split_capital_projects_and_envelopes",
        "wait_for": OUT_03,
        "run": "python data/etl_pipeline/_04_split_capital_projects_and_envelopes.py",
        "expect": OUT_04,
    },
]

# Ensure output directory exists
Path("data_approved/outputs").mkdir(parents=True, exist_ok=True)

def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def wait_for_file(path: Path, poll_seconds=5, timeout_seconds=None):
    start = time.time()
    while True:
        if path.exists() and path.is_file():
            try:
                size = path.stat().st_size
            except OSError:
                size = -1
            if size != 0:
                log(f"Detected file: {path} (size={size} bytes)")
                return True
            log(f"File exists but empty, waiting: {path}")
        else:
            log(f"Waiting for: {path}")

        if timeout_seconds is not None and (time.time() - start) > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for file: {path}")

        time.sleep(poll_seconds)

def run_command(cmd: str):
    log(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed (exit={result.returncode}): {cmd}")

# ---------------------------
# Orchestration
# ---------------------------
log("ETL watcher started.")
for step in PIPELINE:
    log(f"Step {step['name']}: waiting for dependency")
    wait_for_file(step["wait_for"], poll_seconds=POLL_SECONDS, timeout_seconds=TIMEOUT_SECONDS)

    # Optional skip: if expected output already exists and non-empty, skip re-run
    if step["expect"].exists() and step["expect"].stat().st_size > 0:
        log(f"Output already exists, skipping step: {step['expect']}")
        continue

    run_command(step["run"])

    # Validate expected output from this step
    if not step["expect"].exists() or step["expect"].stat().st_size == 0:
        raise FileNotFoundError(f"Expected output not created: {step['expect']}")

    log(f"Completed {step['name']} -> {step['expect']}")

log(f"Pipeline complete. Final file: {OUT_04}")

In [ ]:
from google.colab import files
files.download("data_approved/outputs/budget_split_into_capital_and_all_envelopes.xlsx")